# Mini EDA Project — Week 4

This notebook demonstrates a full exploratory data analysis (EDA) workflow: load → understand → clean → explore → visualize → summarize.

**To use with your own dataset:** in the "Load the data" cell below, comment out the synthetic data block and uncomment the `pd.read_csv(...)` line, pointing it at your own file. Everything after that cell will work the same way regardless of dataset, as long as you adjust the column names referenced later on.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## Step 1 — Load the data

In [ ]:
# --- OPTION A: your own dataset ---
# df = pd.read_csv("your_file.csv")

# --- OPTION B: synthetic demo dataset (comment this whole block out if using Option A) ---
rng = np.random.default_rng(42)
n = 500

cities_clean = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"]
city_variants = {
    "New York": ["New York", "new york", "NEW YORK", " New York"],
    "Los Angeles": ["Los Angeles", "los angeles", "LOS ANGELES"],
    "Chicago": ["Chicago", "chicago"],
    "Houston": ["Houston", "houston "],
    "Phoenix": ["Phoenix", "PHOENIX"],
}

rows = []
for i in range(n):
    true_city = rng.choice(cities_clean)
    city_val = rng.choice(city_variants[true_city])
    age = rng.integers(18, 70)
    category = rng.choice(["Electronics", "Clothing", "Home", "Books", "Sports"])
    amount = round(float(rng.gamma(shape=2.0, scale=40)), 2)
    rating = rng.integers(1, 6)
    signup_date = pd.Timestamp("2023-01-01") + pd.Timedelta(days=int(rng.integers(0, 700)))
    rows.append([i, age, city_val, category, amount, rating, signup_date.strftime("%Y-%m-%d")])

df = pd.DataFrame(rows, columns=[
    "order_id", "customer_age", "city", "product_category",
    "order_amount", "rating", "signup_date"
])

# inject messiness so the cleaning steps below have something to do
missing_age_idx = rng.choice(df.index, size=25, replace=False)
df.loc[missing_age_idx, "customer_age"] = np.nan

missing_rating_idx = rng.choice(df.index, size=15, replace=False)
df.loc[missing_rating_idx, "rating"] = np.nan

outlier_idx = rng.choice(df.index, size=6, replace=False)
df.loc[outlier_idx, "customer_age"] = rng.choice([150, 200, -5], size=6)

negative_amount_idx = rng.choice(df.index, size=4, replace=False)
df.loc[negative_amount_idx, "order_amount"] = -df.loc[negative_amount_idx, "order_amount"]

duplicate_rows = df.sample(10, random_state=1)
df = pd.concat([df, duplicate_rows], ignore_index=True)

df.head()


## Step 2 — Understand the data before touching it

Get an overview before changing anything: shape, column types, and a first look at the values.

In [ ]:
print("Shape:", df.shape)
print()
print(df.dtypes)


In [ ]:
df.info()


In [ ]:
df.describe(include="all")


## Step 3 — Clean the data

Handle missing values, duplicates, inconsistent formatting, wrong types, and clear data-entry errors (outliers that are actually invalid, like a negative price or an age of 200).

Document *why* you made each choice as you go — that reasoning is part of the deliverable.

In [ ]:
# Missing values: how much, and where?
df.isna().sum()


In [ ]:
# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)


In [ ]:
# Standardize inconsistent text formatting
df["city"] = df["city"].str.strip().str.title()
df["city"].value_counts()


In [ ]:
# Fix data types
df["signup_date"] = pd.to_datetime(df["signup_date"])


In [ ]:
# Handle clearly invalid values (data-entry errors, not real outliers)
df.loc[(df["customer_age"] < 0) | (df["customer_age"] > 100), "customer_age"] = np.nan
df.loc[df["order_amount"] < 0, "order_amount"] = np.nan

# Decide how to handle missing values — here, drop rows missing the core numeric fields
# (an alternative would be to impute with median/mode — note your reasoning either way)
df = df.dropna(subset=["customer_age", "order_amount"]).reset_index(drop=True)

df.isna().sum()


## Step 4 — Explore individual variables (univariate)

Look at each column on its own: distributions for numeric columns, counts for categorical columns.

In [ ]:
df[["customer_age", "order_amount", "rating"]].describe()


In [ ]:
df["product_category"].value_counts()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["customer_age"], bins=20, ax=axes[0])
axes[0].set_title("Distribution of Customer Age")
sns.histplot(df["order_amount"], bins=20, ax=axes[1])
axes[1].set_title("Distribution of Order Amount")
plt.tight_layout()
plt.show()


## Step 5 — Explore relationships between variables (bivariate/multivariate)

Look for patterns between variables — does order amount vary by category? Is age related to rating?

In [ ]:
df.groupby("product_category")["order_amount"].mean().sort_values(ascending=False)


In [ ]:
numeric_cols = ["customer_age", "order_amount", "rating"]
corr = df[numeric_cols].corr()
corr


In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Between Numeric Variables")
plt.tight_layout()
plt.show()


## Step 6 — Visualize the key findings

Pick a handful of charts that each answer a specific question — not one of every possible chart type.

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="product_category", y="order_amount")
plt.title("Order Amount by Product Category")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="city", order=df["city"].value_counts().index)
plt.title("Orders by City")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df, x="customer_age", y="order_amount", hue="product_category", alpha=0.6)
plt.title("Order Amount vs Customer Age")
plt.tight_layout()
plt.show()


## Step 7 — Summary of findings

Write a few paragraphs in your own words. Prompts to answer:

- **What is this dataset about?** (what does each row represent, what are the key columns)
- **What did you find interesting or surprising?**
- **What patterns or relationships stood out** in the univariate/bivariate exploration?
- **What limitations or open questions remain** — e.g. data you wish you had, cleaning decisions that could've gone another way?

*(Replace this cell with your own written summary once your exploration is done.)*
